# Memorization and cross-model agreement on MNIST
Train two U-Nets on disjoint subsets of digit 8, then compare deterministic 50-step DDIM samples from matched initial noise seeds.

For each generated image, M is its maximum cosine similarity to its model's training images, averaged across models A and B and five evaluation seeds. C is the mean cosine similarity between A and B outputs from the same seeds. Error bars are population standard deviations across those five sampling seeds (numpy ddof=0), not uncertainty over independently repeated training runs.

These are pixel-space diagnostics: common background pixels affect cosine similarity; the training-bank maximum also depends on bank size. High C and lower M suggest shared structure but do not prove non-copying or a thermodynamic phase transition. The seed-to-endpoint mapping also depends on model initialization and training randomness.

Default QUICK_TEST runs two small dataset sizes for 600 updates per model to check the workflow. Set QUICK_TEST=False for the full sweep. The historical gallery uses earlier full runs; it is not output from this smoke test. No checkpoints are included.


In [ ]:
# ============================== RUN OPTIONS ==============================
RUN_SWEEP  = True     # train missing large-n model pairs (the only GPU-heavy part)
RUN_EVAL   = True     # all figures — cheap, works with whatever checkpoints exist
RUN_BASINS = False     # bonus basin-of-attraction experiment (eval-only, ~5-10 min)
QUICK_TEST = True    # ~10 min smoke test (tiny n, few steps)

NEW_NS           = [250, 500, 1000, 2000]  # the gap; n<=100 comes from old records.json
CKPT_EVERY_MIN   = 15     # wall-clock safety checkpoint to Drive
TIME_BUDGET_MIN  = 210    # stop LAUNCHING trainings after this (everything resumes tomorrow)
EVAL_RESERVE_MIN = 30     # always keep this much budget for the figures
USE_AMP          = True   # mixed precision: ~1.5-2x faster training on T4
REQUIRE_DRIVE    = True   # hard-fail if Drive is not truly writable (no silent /content fallback)
USE_PREV_TUNING  = False  # the standard MNIST-DDPM config is known; old tuning json only if you insist

## Setup and checkpoint storage
In Colab, mount Drive to keep checkpoints between sessions. Elsewhere, MEMGEN_CKPT_DIR selects the checkpoint directory (default ./checkpoints/memgen). Use a fresh directory to keep smoke-test checkpoints separate from full-run checkpoints.

In [ ]:
import os, math, time, random, json, glob, re, zipfile, copy
from dataclasses import dataclass
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

T_START = time.time()
def minutes_elapsed(): return (time.time() - T_START) / 60.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
print("Device:", DEVICE)
if DEVICE == "cpu": print("WARNING: no GPU. Runtime > Change runtime type > GPU.")

def _is_colab():
    try: import google.colab; return True
    except Exception: return False

# ---- Drive mount with WRITE VERIFICATION: never silently fall back to /content ----
CKPT_DIR = os.environ.get("MEMGEN_CKPT_DIR", "./checkpoints/memgen")
if _is_colab():
    mounted = False
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        mounted = os.path.isdir("/content/drive/MyDrive")
    except Exception as e:
        print("Drive mount raised:", e)
    if mounted:
        CKPT_DIR = "/content/drive/MyDrive/memgen_public_smoke" if QUICK_TEST else "/content/drive/MyDrive/memgen_public_full"
    elif REQUIRE_DRIVE:
        raise RuntimeError("Google Drive NOT mounted — refusing to train into volatile /content. "
                           "Re-run this cell (auth popup) or restart the runtime.")
if not _is_colab(): CKPT_DIR = os.path.join(CKPT_DIR, "smoke" if QUICK_TEST else "full")
os.makedirs(CKPT_DIR, exist_ok=True)
_tp = os.path.join(CKPT_DIR, ".write_test")
with open(_tp, "w") as f: f.write(str(time.time()))
os.remove(_tp)   # raises if Drive is mounted read-only / broken

def _path(name): return os.path.join(CKPT_DIR, name)
_existing = sorted(os.path.basename(p) for p in glob.glob(_path("*")))
print(f"Checkpoint dir verified writable -> {CKPT_DIR}   ({len(_existing)} existing files)")
if _existing: print("  found e.g.:", _existing[:10])

def atomic_save(obj, path):
    tmp = path + ".tmp"; torch.save(obj, tmp); os.replace(tmp, path)

def save_json(name, obj):
    tmp = _path(name) + ".tmp"
    with open(tmp, "w") as f: json.dump(obj, f, indent=2)
    os.replace(tmp, _path(name))

def load_json(name, default):
    p = _path(name)
    if not os.path.exists(p): return default
    try:
        with open(p) as f: return json.load(f)
    except Exception as e:
        print(f"WARNING: {name} unreadable ({e}) -> default"); return default

def browser_download(path):
    if _is_colab():
        try:
            from google.colab import files; files.download(path)
        except Exception as e: print("download skipped:", e)

## Diffusion model
The architecture matches the conditional DDPM notebook. The sampler here uses deterministic DDIM rather than the stochastic 1000-step DDPM sampler.

In [ ]:
T = 1000; NUM_CLASSES = 10; NULL_CLASS = NUM_CLASSES; TIME_DIM = 256; EMA_DECAY = 0.999
betas = torch.linspace(1e-4, 0.02, T); alphas = 1 - betas
alphabars = torch.cumprod(alphas, 0)
sqrt_alphabars = torch.sqrt(alphabars); sqrt_one_minus_ab = torch.sqrt(1 - alphabars)
for _n in ["betas", "alphas", "alphabars", "sqrt_alphabars", "sqrt_one_minus_ab"]:
    globals()[_n] = globals()[_n].to(DEVICE)

def extract(a, t, shape): return a.gather(0, t).reshape(t.shape[0], *((1,)*(len(shape)-1)))
def q_sample(x0, t, noise):
    return extract(sqrt_alphabars, t, x0.shape)*x0 + extract(sqrt_one_minus_ab, t, x0.shape)*noise

class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim): super().__init__(); self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        f = torch.exp(-math.log(10000)*torch.arange(half, device=t.device)/(half-1))
        a = t[:, None].float()*f[None]; return torch.cat([torch.sin(a), torch.cos(a)], -1)

class ResBlock(nn.Module):
    def __init__(self, ic, oc, ed, g=8):
        super().__init__()
        self.mlp = nn.Linear(ed, oc); self.c1 = nn.Conv2d(ic, oc, 3, padding=1); self.n1 = nn.GroupNorm(g, oc)
        self.c2 = nn.Conv2d(oc, oc, 3, padding=1); self.n2 = nn.GroupNorm(g, oc)
        self.sk = nn.Conv2d(ic, oc, 1) if ic != oc else nn.Identity()
    def forward(self, x, e):
        h = F.silu(self.n1(self.c1(x))); h = h + self.mlp(F.silu(e))[:, :, None, None]
        h = F.silu(self.n2(self.c2(h))); return h + self.sk(x)

class Down(nn.Module):
    def __init__(self, ic, oc): super().__init__(); self.op = nn.Conv2d(ic, oc, 4, 2, 1)
    def forward(self, x): return self.op(x)

class Up(nn.Module):
    def __init__(self, ic, oc):
        super().__init__(); self.op = nn.Sequential(nn.Upsample(scale_factor=2, mode="nearest"),
                                                    nn.Conv2d(ic, oc, 3, padding=1))
    def forward(self, x): return self.op(x)

class CondUNet(nn.Module):
    def __init__(self, base=64, time_dim=TIME_DIM, num_classes=NUM_CLASSES):
        super().__init__()
        self.time_mlp = nn.Sequential(SinusoidalPosEmb(base), nn.Linear(base, time_dim),
                                      nn.SiLU(), nn.Linear(time_dim, time_dim))
        self.class_emb = nn.Embedding(num_classes+1, time_dim)
        self.init_conv = nn.Conv2d(1, base, 3, padding=1)
        self.d1 = ResBlock(base, base, time_dim);       self.dn1 = Down(base, base*2)
        self.d2 = ResBlock(base*2, base*2, time_dim);   self.dn2 = Down(base*2, base*4)
        self.d3 = ResBlock(base*4, base*4, time_dim);   self.mid = ResBlock(base*4, base*4, time_dim)
        self.up2 = Up(base*4, base*2); self.u2 = ResBlock(base*4, base*2, time_dim)
        self.up1 = Up(base*2, base);   self.u1 = ResBlock(base*2, base, time_dim)
        self.out = nn.Sequential(nn.GroupNorm(8, base), nn.SiLU(), nn.Conv2d(base, 1, 3, padding=1))
    def forward(self, x, t, label):
        e = self.time_mlp(t) + self.class_emb(label)
        h0 = self.init_conv(x); h1 = self.d1(h0, e); h = self.dn1(h1)
        h2 = self.d2(h, e); h = self.dn2(h2); h3 = self.d3(h, e); h = self.mid(h3, e)
        h = self.up2(h); h = self.u2(torch.cat([h, h2], 1), e)
        h = self.up1(h); h = self.u1(torch.cat([h, h1], 1), e)
        return self.out(h)

def copy_model(m):
    m = copy.deepcopy(m).eval()
    for p in m.parameters(): p.requires_grad_(False)
    return m

class EMA:
    def __init__(self, model, decay=EMA_DECAY): self.decay = decay; self.ema = copy_model(model)
    @torch.no_grad()
    def update(self, model):
        for e, p in zip(self.ema.parameters(), model.parameters()):
            e.mul_(self.decay).add_(p, alpha=1-self.decay)
        for eb, b in zip(self.ema.buffers(), model.buffers()): eb.copy_(b)

## Disjoint subsets, evaluation seeds, and DDIM
The fixed data shuffle reserves 300 images and partitions the remaining digit-8 pool into disjoint A/B subsets for each n. Across different n values, subsets can overlap. The reserved images are not used to claim a held-out generalization score.

In [ ]:
tf = transforms.Compose([transforms.Pad(2), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_ds = datasets.MNIST("./data", train=True, download=True, transform=tf)

@dataclass
class Config:
    base_ch: int = 64; lr: float = 2e-4; train_steps: int = 6000
    batch_size: int = 128; guidance_scale: float = 2.0; ddim_steps: int = 50; p_uncond: float = 0.1

TARGET_DIGIT = 8
_idx = (train_ds.targets == TARGET_DIGIT).nonzero(as_tuple=True)[0].tolist()
random.Random(0).shuffle(_idx)                      # same seed as before -> identical subsets
VAL_IDX = _idx[:300]; POOL_IDX = _idx[300:]
print(f"'{TARGET_DIGIT}' pool: {len(POOL_IDX)}  (max n per disjoint pair: {len(POOL_IDX)//2})")

def first_subset(n): return Subset(train_ds, POOL_IDX[:n])
def disjoint_subsets(n, offset=0):
    assert offset + 2*n <= len(POOL_IDX), f"n={n} too big (max {len(POOL_IDX)//2})"
    return Subset(train_ds, POOL_IDX[offset:offset+n]), Subset(train_ds, POOL_IDX[offset+n:offset+2*n])

def job_subset(job_id):
    m = re.match(r"n(\d+)_([AB])$", job_id)
    if m:
        A, B = disjoint_subsets(int(m.group(1))); return A if m.group(2) == "A" else B
    m = re.match(r"evo_n(\d+)$", job_id)
    if m: return first_subset(int(m.group(1)))
    raise ValueError(job_id)

def build_bank(subset):
    M = torch.stack([img.flatten() for img, _ in subset]); M = M / M.norm(dim=1, keepdim=True)
    return M.to(DEVICE), torch.stack([img for img, _ in subset])
def nearest(sample, bank):
    M, imgs = bank; v = sample.flatten().to(DEVICE); v = v / v.norm()
    sims = M @ v; j = int(sims.argmax()); return float(sims[j]), imgs[j]
def cosine(a, b):
    a = a.flatten().float().cpu(); b = b.flatten().float().cpu()
    return torch.dot(a/a.norm(), b/b.norm()).item()

def noise_for_seed(seed, sz=32):
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    return torch.randn(1, 1, sz, sz, generator=g, device=DEVICE)
def noise_for_seeds(seeds): return torch.cat([noise_for_seed(s) for s in seeds], 0)

def slerp(z1, z2, frac):
    a = z1.flatten(); b = z2.flatten()
    omega = torch.acos(torch.clamp((a/a.norm()) @ (b/b.norm()), -1, 1)); so = torch.sin(omega)
    z = (torch.sin((1-frac)*omega)/so)*a + (torch.sin(frac*omega)/so)*b
    return z.reshape(z1.shape)

@torch.no_grad()
def ddim_sample(model, labels, z, cfg, t_start=None):
    """Deterministic DDIM. t_start=None -> from pure noise; t_start=int -> from a given x_{t_start}.
    Classifier-free guidance batched into a single forward pass (2x faster than separate calls)."""
    model.eval(); n = labels.shape[0]; x = z.to(DEVICE)
    null = torch.full((n,), NULL_CLASS, device=DEVICE, dtype=torch.long)
    hi = (T - 1) if t_start is None else int(t_start)
    k = cfg.ddim_steps if t_start is None else max(2, int(round(cfg.ddim_steps * (hi+1) / T)))
    seq = list(reversed(torch.linspace(0, hi, k, dtype=torch.long).tolist()))
    for idx, i in enumerate(seq):
        t = torch.full((n,), int(i), device=DEVICE, dtype=torch.long)
        if cfg.guidance_scale == 1:
            eps = model(x, t, labels)
        else:
            e_c, e_u = model(torch.cat([x, x]), torch.cat([t, t]), torch.cat([labels, null])).chunk(2)
            eps = e_u + cfg.guidance_scale * (e_c - e_u)
        a_t = alphabars[int(i)]
        ip = seq[idx+1] if idx+1 < len(seq) else -1
        a_p = alphabars[int(ip)] if ip >= 0 else torch.tensor(1.0, device=DEVICE)
        x0 = (x - torch.sqrt(1 - a_t)*eps) / torch.sqrt(a_t)
        x = torch.sqrt(a_p)*x0 + torch.sqrt(1 - a_p)*eps
    return x.clamp(-1, 1)

def gen_batch(model, cfg, seeds):
    labels = torch.full((len(seeds),), TARGET_DIGIT, device=DEVICE, dtype=torch.long)
    return ddim_sample(model, labels, noise_for_seeds(seeds), cfg)

def to_img(t): return ((t + 1) / 2).clamp(0, 1).cpu().numpy()

# Configuration used in this experiment; no hyperparameter search in this notebook.
if QUICK_TEST:
    NEW_NS = [10, 50]
    TIME_BUDGET_MIN = 30
    EVAL_RESERVE_MIN = 5
CFG = Config()
if USE_PREV_TUNING:
    _tun = load_json("tuning_results.json", [])
    if _tun:
        CFG = Config(**sorted(_tun, key=lambda d: d["vloss"])[0]["cfg"])
        print("Overriding with previously tuned config:", CFG)
if QUICK_TEST: CFG = Config(**{**vars(CFG), "train_steps": 600})
print("Config:", CFG)

def snap_grid(total, k=8, first=50):
    s = np.unique(np.round(np.logspace(np.log10(first), np.log10(total), k)).astype(int))
    return [int(v) for v in s if v <= total]
SNAP_STEPS = snap_grid(CFG.train_steps, k=8, first=30 if QUICK_TEST else 50)
EVAL_SEEDS = [42, 7, 1234, 8960, 555, 101, 202, 303]
THRESH_MEM, THRESH_GEN = 0.92, 0.90
print("Snapshot steps:", SNAP_STEPS)

## Resumable training
Training saves EMA snapshots and periodic optimizer checkpoints. Resumption continues updates, but does not reproduce the exact random sequence of an uninterrupted run. At n<128 each batch uses n images; with larger n the final batch of an epoch may be smaller than 128.

In [ ]:
def snap_path(job_id, s): return _path(f"snap_{job_id}_{s}.pt")
def _half_sd(sd): return {k: (v.half() if v.is_floating_point() else v) for k, v in sd.items()}

try:
    _scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE == "cuda"))
    def _autocast(): return torch.amp.autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda"))
except Exception:
    _scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and DEVICE == "cuda"))
    def _autocast(): return torch.cuda.amp.autocast(enabled=(USE_AMP and DEVICE == "cuda"))

EST_TRAIN_MIN = 3.0 if QUICK_TEST else 16.0   # per-job estimate; updated with measured throughput after the first job
def can_train(k_jobs=1):
    rem = TIME_BUDGET_MIN - EVAL_RESERVE_MIN - minutes_elapsed()
    ok = rem >= k_jobs * EST_TRAIN_MIN
    if not ok:
        print(f"BUDGET GUARD: {rem:.0f} min left < {k_jobs}x{EST_TRAIN_MIN:.0f} min "
              f"-> not launching. Re-run tomorrow; everything resumes automatically.")
    return ok

def train_model(subset, cfg, job_id, snap_steps=None):
    global EST_TRAIN_MIN
    final_p = _path(f"{job_id}.final.pt"); prog_p = _path(f"{job_id}.inprogress.pt")
    snaps = set(snap_steps or [])
    snaps_missing = [s for s in snaps if not os.path.exists(snap_path(job_id, s))]
    if os.path.exists(final_p) and not snaps_missing:
        print(f"[{job_id}] already complete — skipping"); return True
    if os.path.exists(final_p) and snaps_missing:
        print(f"[{job_id}] final exists but {len(snaps_missing)} snapshots missing -> retraining to capture them")

    loader = DataLoader(subset, batch_size=min(cfg.batch_size, len(subset)), shuffle=True, drop_last=False)
    def _inf():
        while True:
            for b in loader: yield b
    model = CondUNet(base=cfg.base_ch).to(DEVICE); ema = EMA(model)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr); start = 0
    if os.path.exists(prog_p):
        try:
            ck = torch.load(prog_p, map_location=DEVICE, weights_only=True)
            model.load_state_dict(ck["model"]); ema.ema.load_state_dict(ck["ema"])
            opt.load_state_dict(ck["opt"]); start = ck["step"]
            print(f"[{job_id}] resumed @ step {start}")
        except Exception as e:
            print(f"[{job_id}] corrupt in-progress ckpt ({e}) -> starting fresh")

    def _safety(s1):
        atomic_save({"model": model.state_dict(), "ema": ema.ema.state_dict(),
                     "opt": opt.state_dict(), "step": s1, "cfg": vars(cfg)}, prog_p)

    model.train(); it = _inf(); t0 = time.time(); t_last = t0
    pbar = tqdm(range(start, cfg.train_steps), initial=start, total=cfg.train_steps, desc=job_id, leave=False)
    for step in pbar:
        x, y = next(it); x, y = x.to(DEVICE), y.to(DEVICE)
        drop = torch.rand(y.shape[0], device=DEVICE) < cfg.p_uncond
        y = torch.where(drop, torch.full_like(y, NULL_CLASS), y)
        t = torch.randint(0, T, (x.shape[0],), device=DEVICE); noise = torch.randn_like(x)
        with _autocast():
            loss = F.mse_loss(model(q_sample(x, t, noise), t, y), noise)
        opt.zero_grad(set_to_none=True)
        _scaler.scale(loss).backward(); _scaler.step(opt); _scaler.update()
        ema.update(model)
        s1 = step + 1
        if s1 in snaps and not os.path.exists(snap_path(job_id, s1)):
            atomic_save({"ema": _half_sd(ema.ema.state_dict()), "cfg": vars(cfg), "step": s1},
                        snap_path(job_id, s1))
        if (time.time() - t_last) >= CKPT_EVERY_MIN * 60:
            _safety(s1); t_last = time.time()
            pbar.set_postfix_str(f"loss {loss.item():.3f} | ckpt@{s1}")
        elif s1 % 50 == 0:
            pbar.set_postfix_str(f"loss {loss.item():.3f}")
    atomic_save({"ema": ema.ema.state_dict(), "cfg": vars(cfg)}, final_p)
    if os.path.exists(prog_p): os.remove(prog_p)
    steps_done = cfg.train_steps - start
    if steps_done > 200:
        per_full = (time.time() - t0)/60.0 * cfg.train_steps/steps_done
        EST_TRAIN_MIN = 0.5*EST_TRAIN_MIN + 0.5*per_full
        print(f"[{job_id}] done in {(time.time()-t0)/60:.1f} min -> est/job now {EST_TRAIN_MIN:.1f} min")
    return True

def load_final(job_id):
    ck = torch.load(_path(f"{job_id}.final.pt"), map_location=DEVICE, weights_only=True)
    m = CondUNet(base=ck["cfg"]["base_ch"]).to(DEVICE); m.load_state_dict(ck["ema"]); m.eval()
    return m, Config(**ck["cfg"])

def load_snapshot(job_id, step):
    ck = torch.load(snap_path(job_id, step), map_location=DEVICE, weights_only=True)
    m = CondUNet(base=ck["cfg"]["base_ch"]).to(DEVICE); m.load_state_dict(ck["ema"]); m.eval()
    return m, Config(**ck["cfg"])

## Measure and run the sweep

In [ ]:
records = load_json("records.json", [])
done_ns = {r["n"] for r in records}
print("Already measured n:", sorted(done_ns) if done_ns else "none")

def measure_at_n(n, eval_seeds=(42, 7, 1234, 8960, 555)):
    """Same protocol/seeds as the old sweep -> points directly comparable."""
    mA, cA = load_final(f"n{n}_A"); mB, cB = load_final(f"n{n}_B")
    bank_A, bank_B = build_bank(disjoint_subsets(n)[0]), build_bank(disjoint_subsets(n)[1])
    iA = gen_batch(mA, cA, list(eval_seeds)); iB = gen_batch(mB, cB, list(eval_seeds))
    cross = [cosine(iA[j], iB[j]) for j in range(len(eval_seeds))]
    train = [0.5*(nearest(iA[j], bank_A)[0] + nearest(iB[j], bank_B)[0]) for j in range(len(eval_seeds))]
    del mA, mB; torch.cuda.empty_cache()
    return dict(n=n, cross_mean=float(np.mean(cross)), cross_std=float(np.std(cross)),
                train_mean=float(np.mean(train)), train_std=float(np.std(train)))


In [ ]:
# ---- paste-in: defines sweep_over and runs small-n first, then large-n ----
def sweep_over(ns_list):
    recs = load_json("records.json", [])
    print("Already measured n:", sorted({r["n"] for r in recs}) or "none")
    for n in ns_list:
        if 2*n > len(POOL_IDX):
            print(f"n={n}: pool too small — skipping"); continue
        A, B = disjoint_subsets(n)
        ok = True
        for tag, sub in (("A", A), ("B", B)):
            job = f"n{n}_{tag}"
            if os.path.exists(_path(f"{job}.final.pt")) and \
               all(os.path.exists(snap_path(job, s)) for s in SNAP_STEPS):
                print(f"[{job}] complete"); continue
            if not can_train(1): ok = False; break
            train_model(sub, CFG, job, snap_steps=SNAP_STEPS)
        if not ok:
            print("Budget guard tripped — rest resumes on the next run."); break
        if \
           os.path.exists(_path(f"n{n}_A.final.pt")) and os.path.exists(_path(f"n{n}_B.final.pt")):
            rec = measure_at_n(n)
            recs = [r for r in recs if r["n"] != n] + [rec]
            recs.sort(key=lambda r: r["n"]); save_json("records.json", recs)
            print(f"n={n:5d}  cross={rec['cross_mean']:.4f}  nearest-train={rec['train_mean']:.4f}")
    print(f"Elapsed: {minutes_elapsed():.0f} min of {TIME_BUDGET_MIN} min budget.")

if RUN_SWEEP:
    sweep_over(NEW_NS if QUICK_TEST else [10, 25, 50, 100, 250, 500, 1000, 2000])


## Compare endpoint diagnostics
A crossing of the two curves is a descriptive comparison of two different cosine statistics. Five sampling seeds and a single A/B pair per n do not identify a critical point.

In [ ]:
def dataset_baseline_similarity(subset, num_pairs=300, seed=0):
    rng = random.Random(seed); imgs = [img for img, _ in subset]
    return float(np.mean([cosine(imgs[i], imgs[j]) for i, j in
                          (rng.sample(range(len(imgs)), 2) for _ in range(num_pairs))]))

if RUN_EVAL:
    records = load_json("records.json", [])
    records.sort(key=lambda r: r["n"])
    if records:
        baseline = dataset_baseline_similarity(Subset(train_ds, POOL_IDX[:1000]))
        ns = [r["n"] for r in records]
        cm = [r["cross_mean"] for r in records]; cs = [r["cross_std"] for r in records]
        tm = [r["train_mean"] for r in records]; ts = [r["train_std"] for r in records]
        fig, ax = plt.subplots(figsize=(7.5, 5))
        ax.errorbar(ns, cm, yerr=cs, marker="o", capsize=3, color="tab:green",
                    label="cross-model agreement C")
        ax.errorbar(ns, tm, yerr=ts, marker="s", capsize=3, color="tab:red",
                    label="nearest-training similarity M")
        ax.axhline(baseline, color="gray", ls="--", lw=1, label=f"two random eights ≈ {baseline:.2f}")
        ax.set_xscale("log"); ax.set_xlabel("training set size n"); ax.set_ylabel("cosine similarity")
        ax.set_title("Memorization and cross-model agreement vs dataset size")
        ax.legend(loc="center right", fontsize=9); ax.grid(alpha=0.3)
        plt.tight_layout(); plt.savefig(_path("fig1_transition_full.png"), dpi=160); plt.show()
    else:
        print("No records yet — run the sweep first.")

## Inspect samples, nearest training images, and interpolation
These visual checks complement the cosine diagnostics. A nearest-neighbor comparison is not proof of novelty, and a smooth interpolation is not proof of manifold topology.

In [ ]:
if RUN_EVAL:
    finals = {os.path.basename(f)[:-9]: f for f in glob.glob(_path("*.final.pt"))}
    n_jobs = []
    for j in finals:
        m = re.match(r"n(\d+)_A$", j)
        if m: n_jobs.append((int(m.group(1)), j))
        m = re.match(r"evo_n(\d+)$", j)
        if m: n_jobs.append((int(m.group(1)), j))
    n_jobs.sort()
    JOB_MEM = n_jobs[0][1] if n_jobs else None      # smallest n available -> memorization side
    JOB_GEN = n_jobs[-1][1] if n_jobs else None     # largest n available  -> generalization side
    pair_final_ns = sorted(n for n, j in n_jobs if j == f"n{n}_A" and f"n{n}_B" in finals)
    print("memorization-side model:", JOB_MEM, "| generalization-side model:", JOB_GEN,
          "| A/B pairs at n =", pair_final_ns)

    SEEDS4 = [42, 7, 1234, 8960]
    if pair_final_ns:
        nmax = pair_final_ns[-1]
        mA, cfgA = load_final(f"n{nmax}_A"); mB, cfgB = load_final(f"n{nmax}_B")
        iA = gen_batch(mA, cfgA, SEEDS4); iB = gen_batch(mB, cfgB, SEEDS4)
        fig, ax = plt.subplots(2, len(SEEDS4), figsize=(1.8*len(SEEDS4), 3.9))
        for j, s in enumerate(SEEDS4):
            ax[0, j].imshow(to_img(iA[j, 0]), cmap="gray"); ax[0, j].axis("off")
            ax[1, j].imshow(to_img(iB[j, 0]), cmap="gray"); ax[1, j].axis("off")
            ax[0, j].set_title(f"seed {s}\nsim {cosine(iA[j], iB[j]):.3f}", fontsize=9)
        ax[0, 0].text(-0.4, 0.5, "Model A", rotation=90, va="center", transform=ax[0, 0].transAxes, weight="bold")
        ax[1, 0].text(-0.4, 0.5, "Model B", rotation=90, va="center", transform=ax[1, 0].transAxes, weight="bold")
        fig.suptitle(f"Disjoint datasets, same seed (n={nmax}) — matched-seed outputs",
                     y=1.04, fontsize=12)
        fig.savefig(_path(f"fig4_disjoint_n{nmax}.png"), dpi=160, bbox_inches="tight"); plt.show()

        # novelty check: generated vs nearest training image
        bank = build_bank(disjoint_subsets(nmax)[0])
        gen8 = gen_batch(mA, cfgA, EVAL_SEEDS)
        k = len(EVAL_SEEDS)
        fig, ax = plt.subplots(2, k, figsize=(1.5*k, 3.6))
        for j in range(k):
            sim, ni = nearest(gen8[j], bank)
            ax[0, j].imshow(to_img(gen8[j, 0]), cmap="gray"); ax[0, j].axis("off")
            ax[0, j].set_title(f"{sim:.2f}", fontsize=8, color=("tab:red" if sim > 0.95 else "tab:green"))
            ax[1, j].imshow(to_img(ni[0]), cmap="gray"); ax[1, j].axis("off")
        ax[0, 0].text(-0.4, 0.5, "generated", rotation=90, va="center", transform=ax[0, 0].transAxes, fontsize=9)
        ax[1, 0].text(-0.4, 0.5, "nearest\ntrain", rotation=90, va="center", transform=ax[1, 0].transAxes, fontsize=9)
        fig.suptitle(f"n={nmax}: generated images and nearest training images", y=1.04)
        fig.savefig(_path(f"fig5_novelty_n{nmax}.png"), dpi=160, bbox_inches="tight"); plt.show()

        # SLERP through the best generalizing model
        z1, z2 = noise_for_seed(1234), noise_for_seed(8960)
        fr = np.linspace(0, 1, 16)
        zs = torch.cat([slerp(z1, z2, float(f)) for f in fr], 0)
        labels = torch.full((len(fr),), TARGET_DIGIT, device=DEVICE, dtype=torch.long)
        strip = ddim_sample(mA, labels, zs, cfgA)
        fig, ax = plt.subplots(1, len(fr), figsize=(0.95*len(fr), 1.3))
        for j in range(len(fr)):
            ax[j].imshow(to_img(strip[j, 0]), cmap="gray"); ax[j].axis("off")
        fig.suptitle(f"DDIM SLERP on the n={nmax} model — latent interpolation", y=1.2)
        fig.savefig(_path(f"fig6_slerp_n{nmax}.png"), dpi=160, bbox_inches="tight"); plt.show()
        del mA, mB; torch.cuda.empty_cache()